# EDA — Henley Passport Index

Exploratory visualisations: top passports, visa requirement distribution, regional patterns, and ranking trends.

In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

%matplotlib inline
sns.set(style='whitegrid')

def find_file(name):
    patterns = [name, os.path.join('.', name), '/kaggle/input/**/' + name]
    for p in patterns:
        matches = glob.glob(p, recursive=True)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"{name} not found. Check dataset path or dataset_sources in kernel metadata.")

files = {}
for fname in ['countries.csv', 'passport_rankings.csv', 'visa_requirements.csv']:
    try:
        path = find_file(fname)
        files[fname] = path
    except FileNotFoundError as e:
        print(e)

countries = pd.read_csv(files['countries.csv']) if 'countries.csv' in files else pd.DataFrame()
rankings = pd.read_csv(files['passport_rankings.csv']) if 'passport_rankings.csv' in files else pd.DataFrame()
visa = pd.read_csv(files['visa_requirements.csv']) if 'visa_requirements.csv' in files else pd.DataFrame()

print('\nShapes:')
print('countries:', countries.shape)
print('rankings:', rankings.shape)
print('visa_requirements:', visa.shape)


In [ ]:
display(countries.head())
display(rankings.head())
display(visa.head())

In [ ]:
# 1) Top 10 passports by rank (best = lowest rank)
if not rankings.empty:
    top10 = rankings.sort_values(['rank']).head(10).copy()
    plt.figure(figsize=(8,6))
    sns.barplot(x='visa_free_count', y='country_name', data=top10, palette='viridis')
    plt.title('Top 10 Passports — Visa-free destinations')
    plt.xlabel('Visa-free / accessible destinations')
    plt.ylabel('Country')
    plt.tight_layout()
    plt.show()
else:
    print('passport_rankings.csv not loaded')

In [ ]:
# 2) Distribution of requirement types (bar)
if 'requirement_type' in visa.columns and not visa.empty:
    req_counts = visa['requirement_type'].value_counts().reset_index()
    req_counts.columns = ['requirement_type','count']
    plt.figure(figsize=(8,5))
    sns.barplot(x='count', y='requirement_type', data=req_counts, palette='magma')
    plt.title('Visa Requirement Type Distribution')
    plt.xlabel('Count')
    plt.ylabel('Requirement Type')
    plt.tight_layout()
    plt.show()
else:
    print('requirement_type column not found or visa_requirements.csv empty')

In [ ]:
# 3) Regional patterns: counts of requirement types per origin region (stacked bar)
if not visa.empty and not countries.empty and 'from_country_code' in visa.columns:
    # normalize country codes in countries
    c = countries.rename(columns={
        'country_code':'code',
        'country_name':'name'
    })
    merged = visa.merge(c[['code','region']], left_on='from_country_code', right_on='code', how='left')
    pivot = merged.pivot_table(index='region', columns='requirement_type', values='from_country_code', aggfunc='count', fill_value=0)
    pivot_sorted = pivot.sort_values(by=pivot.columns.tolist(), ascending=False)
    pivot_sorted.plot(kind='bar', stacked=True, figsize=(10,6), colormap='tab20')
    plt.title('Region → Requirement Type (stacked counts)')
    plt.xlabel('Origin region')
    plt.ylabel('Number of origin→destination pairs')
    plt.legend(title='Requirement Type', bbox_to_anchor=(1.05,1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('Need visa_requirements.csv and countries.csv with matching codes to compute regional patterns')

In [ ]:
# 4) Ranking trends: visa_free_count over years for several countries
if not rankings.empty and {'country_name','country_code','year','visa_free_count'}.issubset(rankings.columns):
    # pick top 5 latest best-ranked countries
    latest_year = rankings['year'].max()
    sample_countries = rankings[rankings['year']==latest_year].nsmallest(5,'rank')['country_code'].tolist()
    trend = rankings[rankings['country_code'].isin(sample_countries)].copy()
    plt.figure(figsize=(10,6))
    sns.lineplot(data=trend, x='year', y='visa_free_count', hue='country_name', marker='o')
    plt.title(f'Visa-free counts over time — sample (top 5 in {latest_year})')
    plt.xlabel('Year')
    plt.ylabel('Visa-free / accessible destinations')
    plt.tight_layout()
    plt.show()
else:
    print('passport_rankings.csv missing required columns or is empty')

Next steps:
- Add interactive country selector (ipywidgets) to view time series per country.
- Map visa-free coverage on a world choropleth (requires ISO mappings and geopandas/plotly).
- Export plots as PNG for inclusion in reports.